# Quantum Identity Authentication — Qiskit Implementation

Circuit-level Qiskit/Aer simulation of the identity-authentication procedure in Qu, Zhang & Zheng (2022), Section 3.2 / Algorithm 1, plus the attack analysis of Section 4.2 (intercept-measure-resend, Entanglement-Measure) and channel noise.

Ground truth is asserted rather than assumed: Table 3 is verified exactly for all 8 (Bell state, basis) cases, the Eq. (20) decoder is derived from the circuits, and the paper's Section 3.3 worked example runs as a regression test. The attack sections use exact `qiskit.quantum_info` calculations where a closed form exists and seeded Monte-Carlo where it does not.

Qubit layout in each pair circuit: `q0` Charlie (C), `q1` travelling qubit (T), `q2` Eve's forward ancilla (EF), `q3` Eve's return ancilla (ER).

In [1]:
%pip install -q "qiskit>=1.0,<3" "qiskit-aer>=0.14,<1" numpy pandas
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, Operator, DensityMatrix, partial_trace, entropy, random_unitary
from qiskit_aer import AerSimulator
try:
    display
except NameError:
    display = print

backend = AerSimulator()
print("Qiskit and Aer are ready.")

/Users/acube/Rajaram/mini-project-qunatum/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


Qiskit and Aer are ready.


## 1. Protocol parameters

In [2]:
N = 20                  # shared-key length in bits
p = 10                  # overhead parameter: p/2 forward-check + p/2 return-check pairs
ERROR_THRESHOLD = 0.0   # Steps 2 and 4a: a check fails if its error rate is greater than this
KEY_THRESHOLD   = 0.0   # Step 4: allowed symbol error rate in key', the paper's "margin of error"
assert N > 0 and N % 2 == 0 and p > 0 and p % 2 == 0
print(f"Initial pairs={N//2 + p}; forward-check={p//2}; return-check={p//2}; key-carrying={N//2}")
print("The paper gives no numeric value for either threshold; both default to 0 here.")

Initial pairs=20; forward-check=5; return-check=5; key-carrying=10
The paper gives no numeric value for either threshold; both default to 0 here.


## 2. Circuit building blocks

* Bell states (paper notation): U+ = Φ+, U− = Φ−, W+ = Ψ+, W− = Ψ−.
* Alice's Eq. (20) gates: 00→I, 01→Z, 10→X, 11→iY (Qiskit's `Y` differs from iY by a global phase only).
* Bell measurement = inverse Bell-preparation (CNOT, then H) followed by Z-basis measurement.
* Channels: `None`; `"imr_z"`, `"imr_x"`, `"imr_rand"` (intercept-measure-resend with a mid-circuit measurement); `("depol", q)`; `("em", U)` with `U` Eve's 4×4 unitary on (travelling qubit, ancilla), the form of Eq. (21).

In [3]:
C, T, EF, ER = 0, 1, 2, 3
BELL_LABELS = ["U+", "U-", "W+", "W-"]          # index = c0 + 2*c1 of the Bell-transform measurement

def bell_prep(qc, c, t, label):
    qc.h(c); qc.cx(c, t)
    if label in ("U-", "W-"): qc.z(c)
    if label in ("W+", "W-"): qc.x(t)

def bell_transform(qc, c, t):                    # inverse of bell_prep: Bell basis -> computational basis
    qc.cx(c, t); qc.h(c)

KEY_GATES = {"00": None, "01": "z", "10": "x", "11": "y"}     # Eq. (20)
def key_gate(qc, qubit, symbol):
    g = KEY_GATES[symbol]
    if g: getattr(qc, g)(qubit)

def key_symbols(key):
    if len(key) % 2 or set(key) - {"0", "1"}: raise ValueError("Expected even-length binary key.")
    return [key[i:i+2] for i in range(0, len(key), 2)]

def random_key(n, rng):
    return "".join(rng.choice(["0", "1"], size=n).tolist())

def apply_channel(qc, channel, anc, eve_bit, rng=None):
    """Send the travelling qubit T through a channel or attacker.

    Returns the basis of any measurement Eve made on T ("Z" or "X"), else None, so the caller
    knows whether the bit sitting in `eve_bit` is a real reading or just an untouched zero.
    """
    if channel is None: return None
    rng = np.random.default_rng() if rng is None else rng     # every channel below needs randomness
    kind = channel if isinstance(channel, str) else channel[0]
    if kind in ("imr_z", "imr_x", "imr_rand"):
        basis = {"imr_z": "Z", "imr_x": "X"}.get(kind) or str(rng.choice(["Z", "X"]))
        if basis == "X": qc.h(T)
        qc.measure(T, eve_bit)                    # Eve measures, the collapsed qubit is the one she resends
        if basis == "X": qc.h(T)
        return basis
    if kind == "depol":
        if rng.random() < channel[1]:
            getattr(qc, str(rng.choice(["x", "y", "z"])))(T)
        return None
    if kind == "em":
        qc.unitary(channel[1], [anc, T])          # qargs are little-endian: matrix index = 2*t + a
        return None
    if kind == "sparse":
        # ("sparse", f, U): Eve applies the strong unitary U to each particle with probability f.
        if rng.random() < channel[1]:
            qc.unitary(channel[2], [anc, T])
        return None
    if kind == "sparse_imr":
        # ("sparse_imr", f): Z-basis intercept-measure-resend on a fraction f of the particles.
        if rng.random() < channel[1]:
            qc.measure(T, eve_bit)
            return "Z"
        return None
    raise ValueError(f"unknown channel {channel!r}")

def pair_circuit(initial, role, fwd=None, ret=None, symbol=None, basis=None, rng=None):
    # One Bell pair through the whole protocol. role: 'fwd' (forward check), 'ret' (return check) or 'auth' (key pair).
    q = QuantumRegister(4, "q")
    ef, er, out = ClassicalRegister(1, "ef"), ClassicalRegister(1, "er"), ClassicalRegister(2, "out")
    qc = QuantumCircuit(q, ef, er, out)
    bell_prep(qc, q[C], q[T], initial)
    eve_fwd = apply_channel(qc, fwd, q[EF], ef[0], rng)             # Step 2: S2 travels to Alice
    eve_ret = None
    if role == "fwd":
        if basis == "X": qc.h(q[T]); qc.h(q[C])
        qc.measure(q[T], out[0]); qc.measure(q[C], out[1])         # out0 = Alice, out1 = Charlie
    else:
        if role == "auth": key_gate(qc, q[T], symbol)               # Step 3
        eve_ret = apply_channel(qc, ret, q[ER], er[0], rng)         # return channel
        bell_transform(qc, q[C], q[T])
        qc.measure(q[C], out[0]); qc.measure(q[T], out[1])         # Step 4: Bell measurement
    qc.metadata = {"eve_fwd_basis": eve_fwd, "eve_ret_basis": eve_ret}
    return qc

def run_circuits_full(circs, seed):
    """Run one batch and keep everything, including the bits Eve read.

    Aer prints the classical registers in reverse declaration order, so the memory string is
    'out er ef'. That ordering is undocumented, hence the assertion.
    """
    res = backend.run(circs, shots=1, memory=True, seed_simulator=int(seed)).result()
    rows = []
    for i, qc in enumerate(circs):
        toks = res.get_memory(i)[0].split()
        assert len(toks) == 3 and len(toks[0]) == 2, f"expected 'out er ef', got {toks!r}"
        out, er, ef = toks
        md = qc.metadata or {}
        rows.append({"bits": (int(out[-1]), int(out[-2])),                 # (bit0, bit1)
                     "eve_fwd": int(ef) if md.get("eve_fwd_basis") else None,
                     "eve_ret": int(er) if md.get("eve_ret_basis") else None,
                     "eve_fwd_basis": md.get("eve_fwd_basis"),
                     "eve_ret_basis": md.get("eve_ret_basis")})
    return rows

def run_circuits(circs, seed):
    return [row["bits"] for row in run_circuits_full(circs, seed)]

def bell_from_bits(bits):                                           # (c0, c1) -> Bell label
    return BELL_LABELS[bits[0] + 2 * bits[1]]

## 3. Table 3: exact verification

The paper's Table 3 (allowed (Charlie, Alice) outcome pairs) is hard-coded and compared with the exact outcome probabilities of the Qiskit circuits for all 8 (Bell state, basis) cases.

In [4]:
TABLE3 = {   # paper Table 3
    ("U+", "Z"): {"00", "11"}, ("U-", "Z"): {"00", "11"},
    ("W+", "Z"): {"01", "10"}, ("W-", "Z"): {"01", "10"},
    ("U+", "X"): {"++", "--"}, ("U-", "X"): {"+-", "-+"},
    ("W+", "X"): {"++", "--"}, ("W-", "X"): {"+-", "-+"},
}
RELATION = {k: ("same" if len(set(next(iter(v)))) == 1 else "opposite") for k, v in TABLE3.items()}

for (bell, basis), allowed in TABLE3.items():
    qc = QuantumCircuit(2); bell_prep(qc, 0, 1, bell)
    if basis == "X": qc.h(0); qc.h(1)
    seen = set()
    for key, pr in Statevector(qc).probabilities_dict().items():    # key = 'q1 q0' = 'Alice Charlie'
        if pr > 1e-9:
            sym = "01" if basis == "Z" else "+-"
            seen.add(sym[int(key[1])] + sym[int(key[0])])            # Charlie + Alice
    assert seen == allowed, (bell, basis, seen, allowed)
print("Qiskit circuits reproduce Table 3 exactly for all 8 (Bell state, basis) cases.")

# Bell measurement identifies each prepared state
for lab in BELL_LABELS:
    assert bell_from_bits(run_circuits([pair_circuit(lab, "ret")], 1)[0]) == lab
print("Bell measurement identifies all four Bell states.")

Qiskit circuits reproduce Table 3 exactly for all 8 (Bell state, basis) cases.
Bell measurement identifies all four Bell states.


## 4. Decoder table (Eq. 20)

Derived exactly from the circuits: for every initial Bell state and key symbol, which Bell state does Charlie obtain?

In [5]:
DECODE = {}
for initial in BELL_LABELS:
    DECODE[initial] = {}
    for symbol in KEY_GATES:
        qc = QuantumCircuit(2); bell_prep(qc, 0, 1, initial); key_gate(qc, 1, symbol); bell_transform(qc, 0, 1)
        pr = Statevector(qc).probabilities([0, 1])
        k = int(np.argmax(pr)); assert abs(pr[k] - 1) < 1e-12
        DECODE[initial][BELL_LABELS[k]] = symbol
    assert len(DECODE[initial]) == 4
    print(initial, "->", DECODE[initial])

# confirm with Aer sampling: all 16 cases
for initial in BELL_LABELS:
    for symbol in KEY_GATES:
        got = bell_from_bits(run_circuits([pair_circuit(initial, "auth", symbol=symbol)], 7)[0])
        assert DECODE[initial][got] == symbol
print("All 16 encode/decode cases confirmed on Aer.")

U+ -> {'U+': '00', 'U-': '01', 'W+': '10', 'W-': '11'}
U- -> {'U-': '00', 'U+': '01', 'W-': '10', 'W+': '11'}
W+ -> {'W+': '00', 'W-': '01', 'U+': '10', 'U-': '11'}
W- -> {'W-': '00', 'W+': '01', 'U-': '10', 'U+': '11'}
All 16 encode/decode cases confirmed on Aer.


## 5. Full Algorithm 1

Pair bookkeeping: N/2+p Bell pairs, split by a random permutation into p/2 forward-check, p/2 return-check and N/2 key-carrying pairs. Channels act on **every** particle in their direction. All pair circuits of one authentication run in a single batched `backend.run`.

In [6]:
@dataclass
class AuthResult:
    accepted: bool
    reason: str
    forward_error: float
    return_error: float
    key_error: float
    original_key: str
    recovered_key: object
    forward_records: list = field(default_factory=list)
    return_records: list = field(default_factory=list)
    eve_records: list = field(default_factory=list)

    @property
    def checks_passed(self):
        """Eve stayed under both eavesdropping thresholds, i.e. raised no alarm in Step 2 or Step 4a.

        This is the metric that matters for Eve, not `accepted`: her goal in Sections 4.2.2 and
        4.2.3 is to steal the key, and losing the final key comparison costs her nothing.
        """
        return self.reason not in ("Forward-channel check failed", "Return-channel check failed")

def authenticate(shared_key, N, p, threshold=0.0, key_threshold=0.0, rng=None,
                 forward_channel=None, return_channel=None, alice_key=None):
    rng = np.random.default_rng() if rng is None else rng
    if N % 2 or p % 2 or N <= 0 or p <= 0 or len(shared_key) != N:
        raise ValueError("N and p must be positive even numbers and len(shared_key) must equal N.")
    if set(shared_key) - {"0", "1"}: raise ValueError("Key must be binary.")
    alice_key = shared_key if alice_key is None else alice_key
    if len(alice_key) != N or set(alice_key) - {"0", "1"}: raise ValueError("Alice's key must be N binary bits.")

    total = N // 2 + p
    initial = [str(rng.choice(BELL_LABELS)) for _ in range(total)]                    # Step 1
    perm = rng.permutation(total)
    fwd_idx  = sorted(int(i) for i in perm[:p//2])
    ret_idx  = sorted(int(i) for i in perm[p//2:p])
    auth_idx = sorted(int(i) for i in perm[p:])
    assert len(auth_idx) == N // 2 == len(key_symbols(alice_key))                     # zip() below truncates silently
    syms = dict(zip(auth_idx, key_symbols(alice_key)))
    bases = {i: str(rng.choice(["Z", "X"])) for i in fwd_idx}

    order, circs = [], []
    for i in fwd_idx:  order.append(("fwd", i));  circs.append(pair_circuit(initial[i], "fwd", forward_channel, None, basis=bases[i], rng=rng))
    for i in ret_idx:  order.append(("ret", i));  circs.append(pair_circuit(initial[i], "ret", forward_channel, return_channel, rng=rng))
    for i in auth_idx: order.append(("auth", i)); circs.append(pair_circuit(initial[i], "auth", forward_channel, return_channel, symbol=syms[i], rng=rng))
    rows = run_circuits_full(circs, rng.integers(2**31 - 1))

    f_rec, r_rec, e_rec, recovered = [], [], [], []
    for (role, i), row in zip(order, rows):
        bits = row["bits"]
        if role == "fwd":
            alice, charlie = bits
            obs = "same" if alice == charlie else "opposite"
            f_rec.append({"pair": i, "bell": initial[i], "basis": bases[i], "Alice": alice, "Charlie": charlie,
                          "observed": obs, "expected": RELATION[(initial[i], bases[i])],
                          "error": obs != RELATION[(initial[i], bases[i])]})
        elif role == "ret":
            m = bell_from_bits(bits)
            r_rec.append({"pair": i, "initial": initial[i], "measured": m, "error": m != initial[i]})
        else:
            recovered.append(DECODE[initial[i]][bell_from_bits(bits)])
            e_rec.append({"pair": i, "symbol": syms[i],                      # what Eve got to see
                          "eve_fwd": row["eve_fwd"], "eve_fwd_basis": row["eve_fwd_basis"],
                          "eve_ret": row["eve_ret"], "eve_ret_basis": row["eve_ret_basis"]})

    f_err = float(np.mean([r["error"] for r in f_rec]))
    if f_err > threshold:                                                           # Step 2 outcome
        return AuthResult(False, "Forward-channel check failed", f_err, np.nan, np.nan,
                          shared_key, None, f_rec, [], e_rec)
    r_err = float(np.mean([r["error"] for r in r_rec]))
    if r_err > threshold:                                                           # Step 4a
        return AuthResult(False, "Return-channel check failed", f_err, r_err, np.nan,
                          shared_key, None, f_rec, r_rec, e_rec)

    # Step 4b. The paper compares key' with key "within a certain margin of error", so this is a
    # rate test on the recovered symbols. One symbol = one Bell measurement = the granularity
    # at which a disturbed pair goes wrong.
    rec_key = "".join(recovered)
    k_err = float(np.mean([a != b for a, b in zip(recovered, key_symbols(shared_key))]))
    ok = k_err <= key_threshold
    if ok:
        reason = "Key matched; Alice authenticated" if k_err == 0 else "Key within tolerance; Alice authenticated"
    else:
        reason = "Recovered key mismatch; reject Alice"
    return AuthResult(ok, reason, f_err, r_err, k_err, shared_key, rec_key, f_rec, r_rec, e_rec)

master = np.random.default_rng(2026)
shared_key = random_key(N, master)
honest = authenticate(shared_key, N, p, ERROR_THRESHOLD, KEY_THRESHOLD, rng=np.random.default_rng(101))
print("Honest run:", honest.accepted, "|", honest.reason)
print("Original key: ", honest.original_key)
print("Recovered key:", honest.recovered_key)
print(f"Forward error: {honest.forward_error} | Return error: {honest.return_error} | Key symbol error: {honest.key_error}")

print("\nForward-check records:"); display(pd.DataFrame(honest.forward_records))
print("Return-check records:");  display(pd.DataFrame(honest.return_records))

Honest run: True | Key matched; Alice authenticated
Original key:  10010000101111101100
Recovered key: 10010000101111101100
Forward error: 0.0 | Return error: 0.0 | Key symbol error: 0.0

Forward-check records:


,pair,bell,basis,Alice,Charlie,observed,expected,error
0,0,U-,X,1,0,opposite,opposite,False
1,3,U-,Z,0,0,same,same,False
2,14,U+,Z,0,0,same,same,False
3,16,U+,Z,1,1,same,same,False
4,17,W-,Z,0,1,opposite,opposite,False


Return-check records:


,pair,initial,measured,error
0,1,W-,W-,False
1,6,U-,U-,False
2,11,W-,W-,False
3,15,U-,U-,False
4,19,U+,U+,False


## 6. Regression test: the paper's worked example (Section 3.3)

Key `10101100010101011110`, all pairs |Φ+>, forward-check positions 2, 5, 8, 12, 17 (bases Z, X, Z, X, Z), return-check positions 4, 7, 10, 15, 18. The paper's ten Bell outcomes must be reproduced on Aer.

In [7]:
key_AB = "10101100010101011110"
paper_ops      = ["U10","U10","U11","U00","U01","U01","U01","U01","U11","U10"]
paper_outcomes = ["W+","W+","W-","U+","U-","U-","U-","U-","W-","W+"]       # Psi+,Psi+,Psi-,Phi+,Phi-,Phi-,Phi-,Phi-,Psi-,Psi+
assert ["U" + s for s in key_symbols(key_AB)] == paper_ops

fw = run_circuits([pair_circuit("U+", "fwd", basis=b) for b in ["Z", "X", "Z", "X", "Z"]], 5)
assert all(a == c for a, c in fw)                                            # U+ is 'same' in both bases (Table 3)
assert all(bell_from_bits(b) == "U+" for b in run_circuits([pair_circuit("U+", "ret") for _ in range(5)], 6))
outs = [bell_from_bits(b) for b in run_circuits([pair_circuit("U+", "auth", symbol=s) for s in key_symbols(key_AB)], 8)]
assert outs == paper_outcomes, outs
rec = "".join(DECODE["U+"][o] for o in outs)
assert rec == key_AB
print("Paper example reproduced on Aer: ops, Bell outcomes and recovered key", rec, "all match.")

Paper example reproduced on Aer: ops, Bell outcomes and recovered key 10101100010101011110 all match.


## 7. Entanglement-Measure (EM) attack — Section 4.2.3, Eqs. (21)–(24)

Eve couples the travelling qubit to her ancilla |E> with a unitary U:

U|0>|E> = a0|0>|E0> + b0|1>|E1>,  U|1>|E> = a1|0>|E2> + b1|1>|E3>   (Eq. 21)

To stay invisible in Step 2, tracing out Eve must leave |0>, |1>, |+>, |−> unchanged (Eq. 22). The paper derives b0 = a1 = 0 and a0|E0> = b1|E3> (Eq. 23): Eve's ancilla is in a product state and learns nothing. This is tested below with `qiskit.quantum_info`.

In [8]:
def ctrl_ry(theta):
    # Eve rotates her ancilla by Ry(theta) iff the travelling qubit is |1>. theta=0: no attack, theta=pi: full copy.
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, c, -s], [0, 0, s, c]], complex)

CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], complex)
I2 = np.eye(2, dtype=complex)

def haar(d, seed):
    return random_unitary(d, seed=seed).data

def eq21_coefficients(U):
    # Matrix index = 2*t + a (t = travelling qubit, a = ancilla); ancilla starts in |0>.
    o0, o1 = U[:, 0].reshape(2, 2), U[:, 2].reshape(2, 2)      # U|0>|0>, U|1>|0>  as [t, a]
    return {"a0|E0>": o0[0], "b0|E1>": o0[1], "a1|E2>": o1[0], "b1|E3>": o1[1]}

def passes_eq22(U, tol=1e-9):
    # Qubit 0 = ancilla, qubit 1 = travelling qubit. Eq. (22): reduced state of T must equal the input state.
    for prep in (lambda qc: None, lambda qc: qc.x(1), lambda qc: qc.h(1), lambda qc: (qc.x(1), qc.h(1))):
        ref = QuantumCircuit(2); prep(ref)
        qc = QuantumCircuit(2); prep(qc); qc.unitary(U, [0, 1])
        rho_in = partial_trace(Statevector(ref), [0]).data
        rho_out = partial_trace(Statevector(qc), [0]).data
        if not np.allclose(rho_in, rho_out, atol=tol): return False
    return True

candidates = {
    "no attack (identity)":                  np.eye(4, dtype=complex),
    "Eve acts only on her ancilla (I x V)":  np.kron(I2, haar(2, 11)),
    "controlled-Ry(pi/4)":                   ctrl_ry(np.pi / 4),
    "controlled-Ry(pi/2)":                   ctrl_ry(np.pi / 2),
    "CNOT (full copy of Z value)":           CNOT,
    "Haar-random 4x4":                       haar(4, 12),
}
print("Eq. (22) satisfied?  (True = invisible to the Step-2 disturbance test)")
for name, U in candidates.items():
    print(f"  {name:40s} {passes_eq22(U)}")

# ---- Eq. (23) over a candidate pool, and why complete positivity forces it -------------------
PAULI = [np.eye(2, dtype=complex), np.array([[0, 1], [1, 0]], complex),
         np.array([[0, -1j], [1j, 0]]), np.array([[1, 0], [0, -1]], complex)]

def pauli_transfer(U):
    """Pauli transfer matrix of Lambda(rho) = tr_anc(U (rho_T (x) |0><0|_anc) U*), i.e. M[k,j] in
    Lambda(P_j) = sum_k M[k,j] P_k. Qubit 0 = ancilla, qubit 1 = T, so the index is 2*t + a."""
    M = np.zeros((4, 4))
    for j, Pj in enumerate(PAULI):
        rho = np.zeros((4, 4), complex)
        for t1 in range(2):
            for t2 in range(2):
                rho[2 * t1, 2 * t2] = Pj[t1, t2]                   # ancilla in |0><0|
        out = U @ rho @ U.conj().T
        red = np.array([[out[0, 0] + out[1, 1], out[0, 2] + out[1, 3]],
                        [out[2, 0] + out[3, 1], out[2, 2] + out[3, 3]]])   # trace out the ancilla
        for k, Pk in enumerate(PAULI):
            M[k, j] = np.real(np.trace(Pk.conj().T @ red) / 2)
    return M

def is_product_after(U, tol=1e-9):
    # Eq. (23): U|u>|E> = a0|u>|E0>, so each input column factorises. Rank 1 <=> product state.
    return all(np.linalg.matrix_rank(U[:, col].reshape(2, 2), tol=tol) == 1 for col in (0, 2))

pool = ([np.eye(4, dtype=complex)]
        + [haar(4, s) for s in range(300)]                     # unconstrained attacks
        + [np.kron(I2, haar(2, 1000 + s)) for s in range(300)])  # attacks that never touch T
eq22_pool = [U for U in pool if passes_eq22(U, tol=1e-8)]
assert eq22_pool, "no candidate satisfied Eq. (22)"
assert all(is_product_after(U) for U in eq22_pool)
assert all(np.allclose(pauli_transfer(U), np.eye(4), atol=1e-7) for U in eq22_pool)

print(f"\nEq. (23) over a pool: {len(eq22_pool)}/{len(pool)} candidate unitaries satisfy Eq. (22), and every")
print("one of them leaves Eve's ancilla in a product state with T and induces the identity channel on T.")
print("\nThat is forced, not luck. Eq. (22) pins three columns of the Pauli transfer matrix: Lambda fixes")
print("I, X and Z. It says nothing about Y, because |0>,|1>,|+>,|-> only span I, X and Z. But a Pauli-")
print("diagonal map with lambda_X = lambda_Z = 1 is completely positive only if 1 - lambda_Z >= |lambda_X - lambda_Y|,")
print("which forces lambda_Y = 1 too. So no physical attack satisfies Eq. (22) and still leaks, and the")
print("paper's Section 4.2.3 derivation for (Step 2) holds in general. Section 7.3 confirms it with chi = 0.")

print("\nPauli transfer diagonal (lambda_I, lambda_X, lambda_Y, lambda_Z) per candidate:")
for name, U in candidates.items():
    print(f"  {name:40s} {np.round(np.diag(pauli_transfer(U)), 4)}   eq22={passes_eq22(U)}")
print("\nCNOT coefficients (Eq. 21):", {k: np.round(v, 3).tolist() for k, v in eq21_coefficients(CNOT).items()})

Eq. (22) satisfied?  (True = invisible to the Step-2 disturbance test)
  no attack (identity)                     True
  Eve acts only on her ancilla (I x V)     True
  controlled-Ry(pi/4)                      False
  controlled-Ry(pi/2)                      False
  CNOT (full copy of Z value)              False
  Haar-random 4x4                          False



Eq. (23) over a pool: 301/601 candidate unitaries satisfy Eq. (22), and every
one of them leaves Eve's ancilla in a product state with T and induces the identity channel on T.

That is forced, not luck. Eq. (22) pins three columns of the Pauli transfer matrix: Lambda fixes
I, X and Z. It says nothing about Y, because |0>,|1>,|+>,|-> only span I, X and Z. But a Pauli-
diagonal map with lambda_X = lambda_Z = 1 is completely positive only if 1 - lambda_Z >= |lambda_X - lambda_Y|,
which forces lambda_Y = 1 too. So no physical attack satisfies Eq. (22) and still leaks, and the
paper's Section 4.2.3 derivation for (Step 2) holds in general. Section 7.3 confirms it with chi = 0.

Pauli transfer diagonal (lambda_I, lambda_X, lambda_Y, lambda_Z) per candidate:
  no attack (identity)                     [1. 1. 1. 1.]   eq22=True
  Eve acts only on her ancilla (I x V)     [1. 1. 1. 1.]   eq22=True
  controlled-Ry(pi/4)                      [1.     0.9239 0.9239 1.    ]   eq22=False
  controlled-

### 7.1 Forward-channel EM attack: detection probability

For `controlled-Ry(θ)` the per-pair forward error rate is (1 − cos(θ/2))/4: zero in the Z basis, (1 − cos(θ/2))/2 in the X basis.

In [9]:
def forward_error_rate(U, n=800, seed=1):
    rr = np.random.default_rng(seed)
    labs = [str(rr.choice(BELL_LABELS)) for _ in range(n)]
    bas = [str(rr.choice(["Z", "X"])) for _ in range(n)]
    outs = run_circuits([pair_circuit(l, "fwd", ("em", U), None, basis=b, rng=rr) for l, b in zip(labs, bas)], seed)
    return float(np.mean([("same" if a == c else "opposite") != RELATION[(l, b)] for (a, c), l, b in zip(outs, labs, bas)]))

print(f"{'theta':>8} | {'simulated':>9} | {'theory (1-cos(t/2))/4':>22}")
for th in [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]:
    sim, th_val = forward_error_rate(ctrl_ry(th)), (1 - np.cos(th / 2)) / 4
    print(f"{th:8.3f} | {sim:9.3f} | {th_val:22.3f}")
    assert abs(sim - th_val) < 0.05

   theta | simulated |  theory (1-cos(t/2))/4
   0.000 |     0.000 |                  0.000


   0.785 |     0.019 |                  0.019
   1.571 |     0.083 |                  0.073


   2.356 |     0.144 |                  0.154


   3.142 |     0.250 |                  0.250


### 7.2 Return-channel EM attack and the paper's "1/4" claim

Exact probability (Statevector) that Charlie still finds the original Bell state after Eve entangles the returning qubit. For `controlled-Ry(θ)` it equals (1 + cos(θ/2))/2.

In [10]:
def p_correct_return(U):
    vals = []
    for b in BELL_LABELS:
        qc = QuantumCircuit(4); bell_prep(qc, C, T, b); qc.unitary(U, [ER, T]); bell_transform(qc, C, T)
        vals.append(Statevector(qc).probabilities([C, T])[BELL_LABELS.index(b)])
    return float(np.mean(vals))

print(f"{'attack':40s} P(correct Bell state)")
for name, U in candidates.items():
    print(f"  {name:38s} {p_correct_return(U):.3f}")
for th in [np.pi/4, np.pi/2]:
    assert abs(p_correct_return(ctrl_ry(th)) - (1 + np.cos(th/2)) / 2) < 1e-9
hv = [p_correct_return(haar(4, s)) for s in range(500)]
print(f"  {'Haar-random (mean of 500)':38s} {np.mean(hv):.3f}   (min {min(hv):.3f}, max {max(hv):.3f})")
print("\nCNOT (strongest copy-style attack) gives 1/2. The Haar-random average is about 1/4, the random-guess level;")
print("the paper's 1/4 matches that average, not a specific attack.")

attack                                   P(correct Bell state)
  no attack (identity)                   1.000
  Eve acts only on her ancilla (I x V)   1.000
  controlled-Ry(pi/4)                    0.962
  controlled-Ry(pi/2)                    0.854
  CNOT (full copy of Z value)            0.500
  Haar-random 4x4                        0.182


  Haar-random (mean of 500)              0.252   (min 0.005, max 0.663)

CNOT (strongest copy-style attack) gives 1/2. The Haar-random average is about 1/4, the random-guess level;
the paper's 1/4 matches that average, not a specific attack.


### 7.3 How much key does Eve learn? (exact, Holevo bound)

For each key symbol, average Eve's ancilla state over the unknown initial Bell state, trace out Charlie and the travelling qubit, and compute the Holevo quantity χ: the most bits about the 2-bit symbol that *any* measurement could extract.

In [11]:
def rho_E(initial, symbol, fwd_U, ret_U):
    qc = QuantumCircuit(4); bell_prep(qc, C, T, initial)
    if fwd_U is not None: qc.unitary(fwd_U, [EF, T])
    key_gate(qc, T, symbol)
    qc.unitary(ret_U, [ER, T])
    return partial_trace(Statevector(qc), [C, T]).data

def holevo(fwd_U, ret_U):
    rhos = [np.mean([rho_E(b, s, fwd_U, ret_U) for b in BELL_LABELS], axis=0) for s in KEY_GATES]
    S = lambda r: entropy(DensityMatrix(r), base=2)
    return float(S(np.mean(rhos, axis=0)) - np.mean([S(r) for r in rhos]))

print("Holevo information about the 2-bit key symbol (bits, max 2):")
print(f"  return channel only, CNOT                     : {holevo(None, CNOT):.4f}")
print(f"  return channel only, Haar-random              : {holevo(None, haar(4, 5)):.4f}")
print(f"  both channels, CNOT + CNOT                    : {holevo(CNOT, CNOT):.4f}")
print(f"  both channels, ctrl-Ry(pi/2) x2               : {holevo(ctrl_ry(np.pi/2), ctrl_ry(np.pi/2)):.4f}")
print(f"  Eq.(22)-safe forward + CNOT return            : {holevo(np.kron(I2, haar(2, 6)), CNOT):.4f}")
assert abs(holevo(CNOT, CNOT) - 1.0) < 1e-6 and abs(holevo(None, CNOT)) < 1e-6

# Eq. (22)-compliant attacks must leak nothing. Check the whole pool from Section 7, not one example.
chis = [holevo(U, U) for U in eq22_pool[:40]]
assert max(abs(c) for c in chis) < 1e-9, f"an Eq.(22)-safe attack leaked {max(chis)} bits"
print(f"\nAll {len(chis)} Eq.(22)-compliant attacks applied to both channels give chi = 0 bits exactly.")
print("Note which channel matters: return-only CNOT also gives 0, because with the initial Bell state")
print("unknown the returning qubit is maximally mixed. Eve needs both channels to learn anything.")

Holevo information about the 2-bit key symbol (bits, max 2):
  return channel only, CNOT                     : 0.0000
  return channel only, Haar-random              : 0.0000
  both channels, CNOT + CNOT                    : 1.0000
  both channels, ctrl-Ry(pi/2) x2               : 0.3905
  Eq.(22)-safe forward + CNOT return            : 0.0000

All 40 Eq.(22)-compliant attacks applied to both channels give chi = 0 bits exactly.
Note which channel matters: return-only CNOT also gives 0, because with the initial Bell state
unknown the returning qubit is maximally mixed. Eve needs both channels to learn anything.


### 7.4 Full protocol under EM attack: information gained vs. detection

Each row runs the complete Algorithm 1 (N=20, p=10, threshold 0) with Eve attacking **both** channels using `controlled-Ry(θ)` (θ = 0: no attack).

In [12]:
def acceptance(trials, base=70_000, **kw):
    """Returns (final accept rate, forward-check pass rate, both-checks pass rate).

    The last one is Eve's metric: if both checks pass she was never noticed, whatever the
    final key comparison says.
    """
    acc = fwd_pass = both_pass = 0
    for t in range(trials):
        rr = np.random.default_rng(base + t)
        r = authenticate(random_key(N, rr), N, p, ERROR_THRESHOLD, KEY_THRESHOLD, rng=rr, **kw)
        acc += r.accepted
        fwd_pass += (r.reason != "Forward-channel check failed")
        both_pass += r.checks_passed
    return acc / trials, fwd_pass / trials, both_pass / trials

TR = 100
print(f"{'theta':>7} | {'Holevo (bits/symbol)':>20} | {'fwd check pass':>14} | {'both checks pass':>16} | {'FINAL ACCEPT':>12}")
for th in [0, np.pi/8, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]:
    U = ctrl_ry(th)
    a, f, bp = acceptance(TR, forward_channel=("em", U), return_channel=("em", U))
    print(f"{th:7.3f} | {holevo(U, U):20.4f} | {f:14.3f} | {bp:16.3f} | {a:12.3f}")

print(f"\nOther EM variants ({TR} trials):")
for label, kw in [("return-only CNOT",                    dict(return_channel=("em", CNOT))),
                  ("Eq.(22)-safe forward + CNOT return",  dict(forward_channel=("em", np.kron(I2, haar(2, 21))), return_channel=("em", CNOT))),
                  ("Haar-random on both channels",        dict(forward_channel=("em", haar(4, 22)), return_channel=("em", haar(4, 23))))]:
    a, f, bp = acceptance(TR, **kw)
    print(f"  {label:38s} fwd pass={f:.3f}  both checks pass={bp:.3f}  FINAL ACCEPT={a:.3f}")

# A weak attack on every particle is not Eve's only option. A strong attack on a few particles
# trades detection for information along a different curve, and the ctrl-Ry sweep above never
# reaches it: Eve keeps the full CNOT but fires it on a fraction f of the particles.
print(f"\nSparse CNOT ({TR} trials): full-strength copying on a fraction f of particles only")
print(f"  {'f':>5} | {'fwd check pass':>14} | {'both checks pass':>16} | {'FINAL ACCEPT':>12} | {'pairs hit both ways':>19}")
for f_att in [0.05, 0.1, 0.2, 0.4, 1.0]:
    ch = ("sparse", f_att, CNOT)
    a, f, bp = acceptance(TR, base=71_000, forward_channel=ch, return_channel=ch)
    print(f"  {f_att:5.2f} | {f:14.3f} | {bp:16.3f} | {a:12.3f} | {f_att**2 * (N//2):19.2f}")

  theta | Holevo (bits/symbol) | fwd check pass | both checks pass | FINAL ACCEPT


  0.000 |               0.0000 |          1.000 |            1.000 |        1.000


  0.393 |               0.0204 |          0.960 |            0.880 |        0.700


  0.785 |               0.0888 |          0.910 |            0.600 |        0.280


  1.571 |               0.3905 |          0.750 |            0.180 |        0.000


  2.356 |               0.7988 |          0.380 |            0.030 |        0.000


... (5 outputs left out)

   0.10 |          0.910 |            0.460 |        0.180 |                0.10


   0.20 |          0.860 |            0.300 |        0.010 |                0.40


   0.40 |          0.680 |            0.110 |        0.000 |                1.60


   1.00 |          0.260 |            0.010 |        0.000 |               10.00


**Reading the results.** Attacks invisible in Step 2 (Eq. 22) give Eve nothing, as the paper derives. Any attack that extracts information also disturbs the pairs, and the stronger it is the closer final acceptance falls to zero. For weak attacks the 5-sample checks are often blind, but Eve also learns very little, and the key comparison catches the disturbance on the key pairs.

## 8. Other negative tests: wrong key, intercept-resend, noise

In [13]:
wrong_key = ("1" if shared_key[0] == "0" else "0") + shared_key[1:]
tests = {
    "Wrong key":              dict(alice_key=wrong_key, seed=202),
    "Forward IMR (Z)":        dict(forward_channel="imr_z", seed=303),
    "Return IMR (Z)":         dict(return_channel="imr_z", seed=404),
    "Return depol q=0.5":     dict(return_channel=("depol", 0.5), seed=505),
    "EM CNOT, both channels": dict(forward_channel=("em", CNOT), return_channel=("em", CNOT), seed=606),
}
for name, kw in tests.items():
    seed = kw.pop("seed")
    r = authenticate(shared_key, N, p, ERROR_THRESHOLD, KEY_THRESHOLD, rng=np.random.default_rng(seed), **kw)
    print(f"{name:24s} accepted={r.accepted!s:5} | checks_passed={r.checks_passed!s:5} | {r.reason}")
    print(f"{'':24s} fwd_err={r.forward_error:.2f}  ret_err={r.return_error}  key_err={r.key_error}")

Wrong key                accepted=False | checks_passed=True  | Recovered key mismatch; reject Alice
                         fwd_err=0.00  ret_err=0.0  key_err=0.1
Forward IMR (Z)          accepted=False | checks_passed=False | Return-channel check failed
                         fwd_err=0.00  ret_err=0.6  key_err=nan
Return IMR (Z)           accepted=False | checks_passed=False | Return-channel check failed
                         fwd_err=0.00  ret_err=0.2  key_err=nan
Return depol q=0.5       accepted=False | checks_passed=False | Return-channel check failed
                         fwd_err=0.00  ret_err=0.2  key_err=nan
EM CNOT, both channels   accepted=False | checks_passed=False | Return-channel check failed
                         fwd_err=0.00  ret_err=0.4  key_err=nan


## 9. Detection statistics: intercept-resend, theory vs. Monte Carlo

For Z-basis intercept-resend the per-sample forward error is **1/4** and the return error is **1/2**. With p/2 = 5 samples a forward check misses Eve with probability (3/4)^5 ≈ 24 %, so "definitely detected" holds only with many samples.

In [14]:
n = 1000; rr = np.random.default_rng(1)
labs = [str(rr.choice(BELL_LABELS)) for _ in range(n)]; bas = [str(rr.choice(["Z", "X"])) for _ in range(n)]
outs = run_circuits([pair_circuit(l, "fwd", "imr_z", None, basis=b, rng=rr) for l, b in zip(labs, bas)], 3)
errs = [("same" if a == c else "opposite") != RELATION[(l, b)] for (a, c), l, b in zip(outs, labs, bas)]
print("Per-sample forward error under IMR(Z): measured", round(float(np.mean(errs)), 3), "(theory 0.25)")
print(f"Theory: P(forward check misses IMR) = (3/4)^{p//2} = {0.75**(p//2):.3f}")

# The 1/2 return-channel rate was prose only; measure it. A Z measurement anywhere on the round trip
# collapses the pair, so Charlie's Bell measurement lands on the right state only half the time.
outs_r = run_circuits([pair_circuit(l, "ret", None, "imr_z", rng=rr) for l in labs], 4)
outs_f = run_circuits([pair_circuit(l, "ret", "imr_z", None, rng=rr) for l in labs], 5)
e_r = float(np.mean([bell_from_bits(b) != l for b, l in zip(outs_r, labs)]))
e_f = float(np.mean([bell_from_bits(b) != l for b, l in zip(outs_f, labs)]))
print(f"Per-sample return error under IMR(Z) on the return leg:  measured {e_r:.3f} (theory 0.50)")
print(f"Per-sample return error under IMR(Z) on the forward leg: measured {e_f:.3f} (theory 0.50)")
assert abs(e_r - 0.5) < 0.05 and abs(e_f - 0.5) < 0.05
print(f"Theory: P(return check misses IMR)  = (1/2)^{p//2} = {0.5**(p//2):.3f}\n")
for label, kw in [("Honest", {}),
                  ("Forward IMR (Z)", dict(forward_channel="imr_z")),
                  ("Forward IMR (random basis)", dict(forward_channel="imr_rand")),
                  ("Return IMR (Z)", dict(return_channel="imr_z")),
                  ("Wrong key", dict(alice_key="0" * N))]:
    a, f, bp = acceptance(TR, base=10_000, **kw)
    print(f"{label:28s} forward-check pass={f:6.3f}   both checks pass={bp:6.3f}   FINAL ACCEPT={a:6.3f}")

Per-sample forward error under IMR(Z): measured 0.263 (theory 0.25)
Theory: P(forward check misses IMR) = (3/4)^5 = 0.237


Per-sample return error under IMR(Z) on the return leg:  measured 0.517 (theory 0.50)
Per-sample return error under IMR(Z) on the forward leg: measured 0.502 (theory 0.50)
Theory: P(return check misses IMR)  = (1/2)^5 = 0.031



Honest                       forward-check pass= 1.000   both checks pass= 1.000   FINAL ACCEPT= 1.000


Forward IMR (Z)              forward-check pass= 0.230   both checks pass= 0.010   FINAL ACCEPT= 0.000


Forward IMR (random basis)   forward-check pass= 0.290   both checks pass= 0.010   FINAL ACCEPT= 0.000


Return IMR (Z)               forward-check pass= 1.000   both checks pass= 0.050   FINAL ACCEPT= 0.000


Wrong key                    forward-check pass= 1.000   both checks pass= 1.000   FINAL ACCEPT= 0.000


### 9.1 What an IMR eavesdropper actually reads off the key

Section 4.2.2 of the paper ends with two claims: Eve "will **deﬁnitely** be detected" and "she will not
be able to obtain **any** key information". The first is a detection claim and Section 9 above already
measures it. The second is an information claim and it is false as stated.

Eve's inference is elementary. She measures the travelling qubit in the Z basis on the way out and
again on the way back. Of Alice's four Eq. (20) gates, `I` and `Z` leave a Z eigenstate alone while `X`
and `iY` flip it, so **a flip means the symbol's first bit is 1** — one bit per doubly-measured pair,
with no uncertainty and no guessing.

What protects the protocol is not Eve's ignorance but the alarm. So the table reports both: the bits
she reads, and whether anyone noticed. The last column is the one that matters for a static key, because
§3.2 Initialization gives each pair of nodes **one** key of length N and the paper specifies no re-keying
or lockout after a failed check — a detected Eve is told to go away and may simply try again.

In [15]:
def imr_leak(trials, forward_channel, return_channel, base, label):
    """Score what Eve reads from her own IMR bits across full authentications."""
    seen = hits = undetected_bits = n_pass = 0
    for t in range(trials):
        rr = np.random.default_rng(base + t)
        r = authenticate(random_key(N, rr), N, p, ERROR_THRESHOLD, KEY_THRESHOLD, rng=rr,
                         forward_channel=forward_channel, return_channel=return_channel)
        got = 0
        for e in r.eve_records:                       # key-carrying pairs only
            if e["eve_fwd_basis"] == "Z" and e["eve_ret_basis"] == "Z":
                guess = "1" if e["eve_fwd"] != e["eve_ret"] else "0"
                seen += 1
                hits += (guess == e["symbol"][0])     # first bit of the 2-bit symbol
                got += 1
        n_pass += r.checks_passed
        undetected_bits += got if r.checks_passed else 0
    return {"attack": label,
            "P(both checks pass)": n_pass / trials,
            "Eve bit accuracy": (hits / seen) if seen else float("nan"),
            "key bits read / run": seen / trials,
            "key bits read / run, undetected": undetected_bits / trials}

TRIALS = 300
rows = [imr_leak(TRIALS, "imr_z", "imr_z", 20_000, "full IMR(Z), both channels")]
for f_att in [0.1, 0.2, 0.4]:
    ch = ("sparse_imr", f_att)
    rows.append(imr_leak(TRIALS, ch, ch, 21_000, f"sparse IMR(Z), f={f_att}"))
leak = pd.DataFrame(rows).set_index("attack")
display(leak.round(4))

acc = leak["Eve bit accuracy"].dropna()
assert (acc > 0.999).all(), f"Eve's inference should be exact, got {acc.to_dict()}"
print(f"Eve's bit inference is exact ({acc.min():.3f}) in every variant: the paper's 'not ... any key")
print("information' is wrong. There are N/2 =", N // 2, "such bits in the key, one per symbol.")
print("\nRuns needed to read a given key symbol's first bit while never raising an alarm,")
print("assuming the same static key is reused (coupon-collector style, 99% confidence):")
for row in rows:
    per_run = row["key bits read / run, undetected"] / (N // 2)      # per-symbol capture rate
    if per_run > 0:
        n_runs = np.log(0.01) / np.log(1 - min(per_run, 0.999))
        print(f"  {row['attack']:30s} {n_runs:8.0f} authentications")
    else:
        print(f"  {row['attack']:30s}   never undetected in {TRIALS} trials")
print("\nThe per-run leak is small, but it is not zero and it does not reset. The protocol needs a")
print("re-keying rule on a failed check; the paper only says the communication is abandoned.")

,P(both checks pass),Eve bit accuracy,key bits read / run,"key bits read / run, undetected"
attack,,,,
"full IMR(Z), both channels",0.0033,1.0,10.0000,0.0333
"sparse IMR(Z), f=0.1",0.5067,1.0,0.1067,0.0533
"sparse IMR(Z), f=0.2",0.2600,1.0,0.4333,0.1000
"sparse IMR(Z), f=0.4",0.0833,1.0,1.8567,0.1233


Eve's bit inference is exact (1.000) in every variant: the paper's 'not ... any key
information' is wrong. There are N/2 = 10 such bits in the key, one per symbol.

Runs needed to read a given key symbol's first bit while never raising an alarm,
assuming the same static key is reused (coupon-collector style, 99% confidence):
  full IMR(Z), both channels         1379 authentications
  sparse IMR(Z), f=0.1                861 authentications
  sparse IMR(Z), f=0.2                458 authentications
  sparse IMR(Z), f=0.4                371 authentications

The per-run leak is small, but it is not zero and it does not reset. The protocol needs a
re-keying rule on a failed check; the paper only says the communication is abandoned.


### Honest users under noise (threshold trade-off)

In [16]:
print("Honest Alice, depolarising noise q on every particle in both directions.")
print("The Step 2/4a check threshold and the Step 4 key tolerance are independent knobs.\n")
print(f"{'q':>6} | {'thr=0 key=0':>12} | {'thr=.2 key=0':>13} | {'thr=.2 key=.2':>14} | {'thr=.2 key=.4':>14}")
GRID = [(0.0, 0.0), (0.2, 0.0), (0.2, 0.2), (0.2, 0.4)]
for q in [0.0, 0.02, 0.05, 0.1]:
    row = []
    for thr, kthr in GRID:
        acc = 0
        for t in range(80):
            rr = np.random.default_rng(50_000 + t)
            acc += authenticate(random_key(N, rr), N, p, thr, kthr, rng=rr,
                                forward_channel=("depol", q), return_channel=("depol", q)).accepted
        row.append(acc / 80)
    print(f"{q:6.2f} | {row[0]:12.3f} | {row[1]:13.3f} | {row[2]:14.3f} | {row[3]:14.3f}")

# Which test is actually doing the rejecting?
print("\nRejection reasons at q=0.02, by threshold pair:")
import collections
for thr, kthr in GRID:
    reasons = collections.Counter()
    for t in range(300):
        rr = np.random.default_rng(900_000 + t)
        r = authenticate(random_key(N, rr), N, p, thr, kthr, rng=rr,
                         forward_channel=("depol", 0.02), return_channel=("depol", 0.02))
        reasons[r.reason.split(";")[0]] += 1
    print(f"  thr={thr}, key={kthr}: " + ", ".join(f"{k}={v}" for k, v in reasons.most_common()))
print("\nOnce the check threshold is loose, neither eavesdropping check fires and the key comparison is")
print("the only test rejecting honest users, so the Step 4 tolerance is what sets the noise budget.")

Honest Alice, depolarising noise q on every particle in both directions.
The Step 2/4a check threshold and the Step 4 key tolerance are independent knobs.

     q |  thr=0 key=0 |  thr=.2 key=0 |  thr=.2 key=.2 |  thr=.2 key=.4


  0.00 |        1.000 |         1.000 |          1.000 |          1.000


  0.02 |        0.425 |         0.625 |          0.988 |          0.988


  0.05 |        0.150 |         0.275 |          0.863 |          0.900


  0.10 |        0.025 |         0.087 |          0.550 |          0.662

Rejection reasons at q=0.02, by threshold pair:


  thr=0.0, key=0.0: Key matched=145, Recovered key mismatch=80, Return-channel check failed=58, Forward-channel check failed=17


  thr=0.2, key=0.0: Key matched=192, Recovered key mismatch=103, Return-channel check failed=5


  thr=0.2, key=0.2: Key matched=192, Key within tolerance=102, Return-channel check failed=5, Recovered key mismatch=1


  thr=0.2, key=0.4: Key matched=192, Key within tolerance=103, Return-channel check failed=5

Once the check threshold is loose, neither eavesdropping check fires and the key comparison is
the only test rejecting honest users, so the Step 4 tolerance is what sets the noise budget.


### The price of that tolerance: partial-knowledge impersonation

A key tolerance is not free. The paper asks for one in Step 4 but never says how large, and the
symbol error rate it bounds is exactly the quantity an impostor with partial key knowledge needs to
stay under. The table below is the other half of the trade-off, and it is the reason `KEY_THRESHOLD`
defaults to 0 here: at a tolerance of 0.2 an impostor who has 19 of the 20 key bits walks straight in.

In [17]:
shared_imp = random_key(N, np.random.default_rng(5))
TOLS = (0.0, 0.1, 0.2, 0.4)
print("Impostor who knows all but k of the N key bits; eavesdropping checks at threshold 0.2.")
print(f"{'wrong bits':>10} | " + " | ".join(f"{'key tol=' + str(kt):>12}" for kt in TOLS))
for wrong in [1, 2, 4]:
    row = []
    for kt in TOLS:
        acc = 0
        for t in range(200):
            rr = np.random.default_rng(600_000 + t)
            guess = list(shared_imp)
            for j in rr.choice(N, size=wrong, replace=False):
                guess[j] = "1" if guess[j] == "0" else "0"
            acc += authenticate(shared_imp, N, p, 0.2, kt, rng=rr, alice_key="".join(guess)).accepted
        row.append(acc / 200)
    print(f"{wrong:10d} | " + " | ".join(f"{v:12.3f}" for v in row))
print("\nk wrong bits corrupt at most k of the N/2 symbols, so any tolerance >= k/(N/2) admits the")
print("impostor. Choosing it is a real design decision the paper leaves open: too tight and honest")
print("nodes fail under channel noise, too loose and partial key knowledge is enough to authenticate.")

Impostor who knows all but k of the N key bits; eavesdropping checks at threshold 0.2.
wrong bits |  key tol=0.0 |  key tol=0.1 |  key tol=0.2 |  key tol=0.4


         1 |        0.000 |        1.000 |        1.000 |        1.000


         2 |        0.000 |        0.060 |        1.000 |        1.000


         4 |        0.000 |        0.000 |        0.005 |        1.000

k wrong bits corrupt at most k of the N/2 symbols, so any tolerance >= k/(N/2) admits the
impostor. Choosing it is a real design decision the paper leaves open: too tight and honest
nodes fail under channel noise, too loose and partial key knowledge is enough to authenticate.


## 10. Representative Qiskit circuits

In [18]:
rr = np.random.default_rng(0)
print("Honest key pair (U+, key symbol 01):")
print(pair_circuit("U+", "auth", symbol="01").draw(output="text", fold=120))
print("\nForward-check pair (W-, X basis):")
print(pair_circuit("W-", "fwd", basis="X").draw(output="text", fold=120))
print("\nIntercept-resend (Z) on both channels, key pair:")
print(pair_circuit("U+", "auth", "imr_z", "imr_z", symbol="01", rng=rr).draw(output="text", fold=120))
print("\nEM attack (CNOT) on both channels, key pair:")
print(pair_circuit("U+", "auth", ("em", CNOT), ("em", CNOT), symbol="01", rng=rr).draw(output="text", fold=120))

Honest key pair (U+, key symbol 01):
       ┌───┐               ┌───┐┌─┐
  q_0: ┤ H ├──■─────────■──┤ H ├┤M├
       └───┘┌─┴─┐┌───┐┌─┴─┐└┬─┬┘└╥┘
  q_1: ─────┤ X ├┤ Z ├┤ X ├─┤M├──╫─
            └───┘└───┘└───┘ └╥┘  ║ 
  q_2: ──────────────────────╫───╫─
                             ║   ║ 
  q_3: ──────────────────────╫───╫─
                             ║   ║ 
 ef: 1/══════════════════════╬═══╬═
                             ║   ║ 
 er: 1/══════════════════════╬═══╬═
                             ║   ║ 
out: 2/══════════════════════╩═══╩═
                             1   0 

Forward-check pair (W-, X basis):
       ┌───┐     ┌───┐┌───┐   ┌─┐
  q_0: ┤ H ├──■──┤ Z ├┤ H ├───┤M├
       └───┘┌─┴─┐├───┤├───┤┌─┐└╥┘
  q_1: ─────┤ X ├┤ X ├┤ H ├┤M├─╫─
            └───┘└───┘└───┘└╥┘ ║ 
  q_2: ─────────────────────╫──╫─
                            ║  ║ 
  q_3: ─────────────────────╫──╫─
                            ║  ║ 
 ef: 1/═════════════════════╬══╬═
                            ║  ║ 
 er: 1/═══════

## 11. Mapping to the paper

| Implementation | Paper reference |
|---|---|
| `bell_prep` for the four Bell states | Eqs. (16)–(19) |
| Charlie keeps C, sends T | Step 2 |
| Z/X forward check, Table 3 verified exactly | Step 2, Table 3 |
| `key_gate`: I, Z, X, iY | Eq. (20), Step 3 |
| `bell_transform` + measurement, `DECODE`, key comparison | Step 4, Algorithm 1 |
| Section 3.3 replay on Aer (assertions) | Tables 3 & 5, Sec. 3.3 |
| `imr_*` channels on every particle | Sec. 4.2.2 |
| `("em", U)` channel, Eq. (22) test, Eq. (23) pool check, Pauli transfer argument, Holevo | Sec. 4.2.3, Eqs. (21)–(24) |
| `key_threshold` on the recovered-symbol error rate | Step 4, "within a certain margin of error" |
| `eve_records`, Section 9.1 bit-recovery score | Sec. 4.2.2, "will not obtain any key information" |

## 12. Limitations
1. Aer is a classical simulator, not physical hardware; ideal unless noise is added explicitly.
2. BB84/B92 key distribution is not implemented; the shared key is an input.
3. Eve's EM attack uses one ancilla per channel and one unitary per run.
4. Monte-Carlo sizes are small to keep the notebook fast; raise `TR` and `TRIALS` for tighter estimates.
5. Authentication module only; one-way authentication with a static key.
6. Alice and Charlie's check qubits are measured in one circuit, so the paper's classical announcement
   step (Alice publishes bases and results, Charlie then measures) is not modelled. Detection statistics
   are unaffected, but an Eve who exploits the announcement cannot be expressed.
7. Eve is assumed unable to target the check positions. That holds for the forward leg; on the return leg
   the paper has Alice "expose the position of the eavesdropping particles", and it does not say she does
   so only after the particles have arrived.